**Purpose:** `v1_model.ipynb` — part of `02_sentiment/reddit` (see the stage README).

**Inputs:** `/kaggle/input/datasets/hugoverssimo/redditsentiment/dataset_augmented.parquet`

**Outputs:** `learning_curve_train_mean_std.csv`, `learning_curve_val_mean_std.csv`, `train_loss_curve.csv`, `val_loss_curve.csv`, `{split_name}__cm_norm.csv`

**Notes:** paths resolve via `src.config` (run `pip install -e .` from the repo root first).


In [ ]:
from src.seeds import SEED_SENTIMENT_REDDIT_V1_MODEL


https://www.kaggle.com/code/hugoverssimo/fork-of-reddit-sentiment

In [ ]:
# ===== Cell 1 (Kaggle-safe) =====
# Fixes NumPy/SciPy/sklearn binary compatibility + keeps your parquet stack stable.

!pip -q uninstall -y numpy scipy scikit-learn pandas pyarrow

# Pin to a known-good combo for Python 3.12 wheels on Kaggle
!pip -q install --no-cache-dir --force-reinstall \
  "numpy==2.0.2" \
  "scipy==1.14.1" \
  "scikit-learn==1.5.2" \
  "pandas==2.2.3" \
  "pyarrow==18.1.0"

!pip -q install flash-attn

# HF stack
!pip -q install -U --no-cache-dir \
  "transformers>=4.48.0,<5.3.0" \
  "accelerate>=0.34.0" \
  datasets evaluate

import numpy, scipy, sklearn, pandas, pyarrow
print("numpy", numpy.__version__)
print("scipy", scipy.__version__)
print("sklearn", sklearn.__version__)
print("pandas", pandas.__version__)
print("pyarrow", pyarrow.__version__)

In [ ]:
import pandas as pd
import json

data_all = pd.read_parquet("/kaggle/input/datasets/hugoverssimo/redditsentiment/dataset_augmented.parquet")

with open("/kaggle/input/datasets/hugoverssimo/redditsentiment/reddit_gpt_validation_final.json", "r") as f:
    reddit_gpt_validation_with_labels = json.load(f)
data_test_ids = list(reddit_gpt_validation_with_labels.keys())

data_test = data_all[data_all.index.isin(data_test_ids)]

data_train = data_all[~data_all.index.isin(data_test_ids)]


train_data_per_sector = {}
for sector in [sec for sec in data_train.columns if sec.startswith("label_")]:
    train_data_per_sector[sector] = dict(data_train[sector].value_counts())
train_data_per_sector = pd.DataFrame(train_data_per_sector).T.fillna(0).astype(int)
train_data_per_sector

In [ ]:
# ============================================================
# ModernBERT Multi-Head (11 sectors) | 5-fold Stratified CV (run folds 0..2)
# - Stratification proxy keeps multi-sector label patterns represented
# - Saves per-sector JSON (train+val) + global reports/CM + learning curves
# - No HF checkpoint saving
# ============================================================

import os
import json
import hashlib
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModel,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)

# -------------------------
# CONFIG
# -------------------------
CONFIG = {
    "seed": SEED_SENTIMENT_REDDIT_V1_MODEL,
    "model_id": "answerdotai/ModernBERT-base",
    "text_col": "thread",
    "sector_cols": [
        "label_Information Technology",
        "label_Health Care",
        "label_Materials",
        "label_Financials",
        "label_Consumer Discretionary",
        "label_Utilities",
        "label_Communication Services",
        "label_Real Estate",
        "label_Consumer Staples",
        "label_Energy",
        "label_Industrials",
    ],
    "sentiments": ["negative", "neutral", "positive"],

    # CV
    "n_folds": 5,
    "folds_to_run": 3,  # run folds 0..(folds_to_run-1)
    #"early_stopping_patience": 3,

    # Trainer logging/eval cadence
    "eval_strategy": "steps",
    "eval_steps": 200,
    "logging_steps": 50,

    # Mixed precision
    "fp16": True,

    # Output
    "output_root": "modernbert_cv",
}

# -------------------------
# Small utils
# -------------------------
def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def save_json(obj: Any, path: str) -> None:
    ensure_dir(os.path.dirname(path))
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)

def save_txt(text: str, path: str) -> None:
    ensure_dir(os.path.dirname(path))
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)

def hparams_hash(hparams: Dict[str, Any], n_chars: int = 10) -> str:
    s = json.dumps(hparams, sort_keys=True, ensure_ascii=True)
    return hashlib.sha1(s.encode("utf-8")).hexdigest()[:n_chars]

# -------------------------
# Stratification proxy
# -------------------------
def make_stratify_keys(y: np.ndarray, n_folds: int, num_classes: int) -> np.ndarray:
    keys = np.array(["-".join(map(str, row.tolist())) for row in y], dtype=object)
    vc = pd.Series(keys).value_counts()

    rare = set(vc[vc < n_folds].index.tolist())
    keys = np.array([k if k not in rare else "RARE" for k in keys], dtype=object)

    # Fallback: mode sentiment across sectors
    if len(pd.Series(keys).unique()) < n_folds:
        mode_proxy = []
        for row in y:
            counts = np.bincount(row, minlength=num_classes)
            mode_proxy.append(int(np.argmax(counts)))
        keys = np.array(mode_proxy, dtype=np.int64)

    return keys

# -------------------------
# Data / weights
# -------------------------
class SectorSentimentDataset(torch.utils.data.Dataset):
    def __init__(self, texts: List[str], labels: np.ndarray, tokenizer, max_len: int):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_len,
            padding=False,
        )
        item = {k: torch.tensor(v, dtype=torch.long) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)  # (num_sectors,)
        return item

def compute_class_weights_per_sector(y_int: np.ndarray, num_classes: int) -> torch.Tensor:
    """
    Inverse-frequency weights per sector. Shape: (num_sectors, num_classes)
    """
    eps = 1e-6
    num_sectors = y_int.shape[1]
    weights = np.zeros((num_sectors, num_classes), dtype=np.float32)

    for s in range(num_sectors):
        counts = np.bincount(y_int[:, s], minlength=num_classes).astype(np.float32)
        total = counts.sum()
        weights[s] = total / (num_classes * (counts + eps))

    return torch.tensor(weights, dtype=torch.float32)

# -------------------------
# Model
# -------------------------
class ModernBERTMultiHead(nn.Module):
    def __init__(self, model_id: str, num_sectors: int, num_classes: int, dropout: float = 0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_id)
        hidden = self.encoder.config.hidden_size
        self.drop = nn.Dropout(dropout)
        self.heads = nn.ModuleList([nn.Linear(hidden, num_classes) for _ in range(num_sectors)])

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, labels=None):
        enc_kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            enc_kwargs["token_type_ids"] = token_type_ids

        out = self.encoder(**enc_kwargs)
        pooled = getattr(out, "pooler_output", None)
        if pooled is None:
            pooled = out.last_hidden_state[:, 0]

        pooled = self.drop(pooled)
        logits = torch.stack([head(pooled) for head in self.heads], dim=1)  # (B, S, C)

        if labels is not None:
            return {"logits": logits, "labels": labels}
        return {"logits": logits}

class WeightedMultiHeadTrainer(Trainer):
    def __init__(self, class_weights: torch.Tensor, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights  # (S, C)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")                 # (B, S)
        outputs = model(**inputs, labels=labels)
        logits = outputs["logits"]                    # (B, S, C)

        loss = 0.0
        for s in range(logits.size(1)):
            w = self.class_weights[s].to(logits.device)
            loss = loss + nn.functional.cross_entropy(logits[:, s, :], labels[:, s], weight=w)

        loss = loss / logits.size(1)
        return (loss, outputs) if return_outputs else loss

# -------------------------
# Metrics + prediction helpers
# -------------------------
def unwrap_logits(preds: Any, num_classes: int) -> np.ndarray:
    if isinstance(preds, (tuple, list)):
        cand = None
        for p in preds:
            if hasattr(p, "shape") and len(p.shape) == 3:
                cand = p
                break
        preds = cand if cand is not None else preds[0]

    preds = np.asarray(preds)
    if preds.ndim == 4 and preds.shape[-1] == num_classes:
        preds = preds[0]
    return preds

def make_compute_metrics(num_sectors: int, num_classes: int):
    def compute_metrics(eval_pred):
        preds, labels = eval_pred
        preds = unwrap_logits(preds, num_classes)
        labels = np.asarray(labels)
        pred_ids = np.argmax(preds, axis=-1)  # (N, S)

        f1s, accs = [], []
        for s in range(num_sectors):
            f1s.append(f1_score(labels[:, s], pred_ids[:, s], average="macro"))
            accs.append(accuracy_score(labels[:, s], pred_ids[:, s]))

        return {
            "macro_f1_mean": float(np.mean(f1s)),
            "acc_mean": float(np.mean(accs)),
        }
    return compute_metrics

@torch.no_grad()
def predict_ids(trainer: Trainer, dataset: torch.utils.data.Dataset, num_classes: int) -> Tuple[np.ndarray, np.ndarray]:
    out = trainer.predict(dataset)
    logits = unwrap_logits(out.predictions, num_classes)
    pred_ids = np.argmax(logits, axis=-1)
    true_ids = np.asarray(out.label_ids)
    return pred_ids, true_ids

# -------------------------
# Saving artifacts
# -------------------------
def flatten_multihead(y_2d: np.ndarray) -> np.ndarray:
    return y_2d.reshape(-1)

def save_classification_report_txt(out_dir: str, split_name: str, y_true: np.ndarray, y_pred: np.ndarray, target_names: List[str]):
    rep_txt = classification_report(y_true, y_pred, target_names=target_names, digits=4, zero_division=0)
    save_txt(rep_txt, os.path.join(out_dir, f"{split_name}__classification_report.txt"))

def save_confusion_matrix_norm_csv(out_dir: str, split_name: str, y_true: np.ndarray, y_pred: np.ndarray, target_names: List[str]):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(target_names))), normalize="true")
    cm_df = pd.DataFrame(cm, index=[f"true_{x}" for x in target_names], columns=[f"pred_{x}" for x in target_names])
    ensure_dir(out_dir)
    cm_df.to_csv(os.path.join(out_dir, f"{split_name}__cm_norm.csv"), index=True)

def save_global_fold_artifacts(fold_dir: str, y_true_train: np.ndarray, y_pred_train: np.ndarray, y_true_val: np.ndarray, y_pred_val: np.ndarray, target_names: List[str]):
    ensure_dir(fold_dir)
    # TRAIN
    tr_true = flatten_multihead(y_true_train)
    tr_pred = flatten_multihead(y_pred_train)
    save_classification_report_txt(fold_dir, "GLOBAL_TRAIN", tr_true, tr_pred, target_names)
    save_confusion_matrix_norm_csv(fold_dir, "GLOBAL_TRAIN", tr_true, tr_pred, target_names)
    # VAL
    va_true = flatten_multihead(y_true_val)
    va_pred = flatten_multihead(y_pred_val)
    save_classification_report_txt(fold_dir, "GLOBAL_VAL", va_true, va_pred, target_names)
    save_confusion_matrix_norm_csv(fold_dir, "GLOBAL_VAL", va_true, va_pred, target_names)

def save_per_sector_train_val_json(out_dir: str, sector_cols: List[str], sentiments: List[str], y_true_train: np.ndarray, y_pred_train: np.ndarray, y_true_val: np.ndarray, y_pred_val: np.ndarray):
    ensure_dir(out_dir)
    num_classes = len(sentiments)

    for s, col in enumerate(sector_cols):
        sector_name = col.replace("label_", "").replace("/", "_")

        def pack(y_true_1d, y_pred_1d):
            rep = classification_report(y_true_1d, y_pred_1d, target_names=sentiments, digits=4, output_dict=True, zero_division=0)
            cm_raw = confusion_matrix(y_true_1d, y_pred_1d, labels=list(range(num_classes)), normalize=None)
            cm_norm = confusion_matrix(y_true_1d, y_pred_1d, labels=list(range(num_classes)), normalize="true")
            return {
                "classification_report": rep,
                "confusion_matrix_raw": cm_raw.tolist(),
                "confusion_matrix_norm_true": cm_norm.tolist(),
                "labels": sentiments,
            }

        payload = {
            "sector": sector_name,
            "train": pack(y_true_train[:, s], y_pred_train[:, s]),
            "val": pack(y_true_val[:, s], y_pred_val[:, s]),
        }
        save_json(payload, os.path.join(out_dir, f"sector_{sector_name}__train_val_metrics.json"))

def extract_loss_curves(log_history: List[Dict[str, Any]]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    train_rows, eval_rows = [], []
    for row in log_history:
        if "loss" in row and "step" in row and "eval_loss" not in row:
            train_rows.append({"step": row["step"], "loss": row["loss"]})
        if "eval_loss" in row and "step" in row:
            eval_rows.append({"step": row["step"], "loss": row["eval_loss"]})
    train_df = pd.DataFrame(train_rows).drop_duplicates("step").sort_values("step")
    eval_df = pd.DataFrame(eval_rows).drop_duplicates("step").sort_values("step")
    return train_df, eval_df

def aggregate_curves(curves: List[pd.DataFrame], curve_name: str) -> pd.DataFrame:
    if not curves:
        return pd.DataFrame(columns=["step", "mean", "std", "n", "curve"])

    steps = sorted(set(np.concatenate([c["step"].values for c in curves if len(c) > 0])))
    if not steps:
        return pd.DataFrame(columns=["step", "mean", "std", "n", "curve"])

    aligned = []
    for c in curves:
        if len(c) == 0:
            continue
        tmp = c.set_index("step").reindex(steps).ffill()
        aligned.append(tmp["loss"].values)

    if not aligned:
        return pd.DataFrame(columns=["step", "mean", "std", "n", "curve"])

    mat = np.vstack(aligned)
    return pd.DataFrame({
        "step": steps,
        "mean": mat.mean(axis=0),
        "std": mat.std(axis=0),
        "n": mat.shape[0],
        "curve": curve_name,
    })

def plot_learning_curve(train_agg: pd.DataFrame, val_agg: pd.DataFrame, out_png: str, title: str):
    plt.figure()
    if len(train_agg) > 0:
        plt.plot(train_agg["step"], train_agg["mean"], label="train_loss_mean")
        plt.fill_between(train_agg["step"], train_agg["mean"] - train_agg["std"], train_agg["mean"] + train_agg["std"], alpha=0.2)
    if len(val_agg) > 0:
        plt.plot(val_agg["step"], val_agg["mean"], label="val_loss_mean")
        plt.fill_between(val_agg["step"], val_agg["mean"] - val_agg["std"], val_agg["mean"] + val_agg["std"], alpha=0.2)
    plt.xlabel("step")
    plt.ylabel("loss")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    ensure_dir(os.path.dirname(out_png))
    plt.savefig(out_png, dpi=150)
    plt.close()

# -------------------------
# CV runner
# -------------------------
def run_cv_once(
    df: pd.DataFrame,
    y_all: np.ndarray,
    texts_all: List[str],
    strat_keys: np.ndarray,
    hparams: Dict[str, Any],
    run_name: Optional[str] = None,
) -> Dict[str, Any]:
    device = "cuda" if torch.cuda.is_available() else "cpu"

    sector_cols = CONFIG["sector_cols"]
    sentiments = [s.lower() for s in CONFIG["sentiments"]]
    num_sectors = len(sector_cols)
    num_classes = len(sentiments)

    tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_id"])
    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    hp_hash = hparams_hash(hparams)
    run_dir = os.path.join(CONFIG["output_root"], run_name or f"run_{hp_hash}")
    ensure_dir(run_dir)

    skf = StratifiedKFold(n_splits=CONFIG["n_folds"], shuffle=True, random_state=CONFIG["seed"])
    folds = list(skf.split(np.arange(len(df)), strat_keys))[: CONFIG["folds_to_run"]]

    compute_metrics = make_compute_metrics(num_sectors, num_classes)

    fold_rows = []
    fold_train_curves, fold_val_curves = [], []

    for fold_i, (train_idx, val_idx) in enumerate(folds):
        fold_dir = os.path.join(run_dir, f"fold_{fold_i}")
        ensure_dir(fold_dir)

        train_texts = [texts_all[i] for i in train_idx]
        val_texts = [texts_all[i] for i in val_idx]
        y_train = y_all[train_idx]
        y_val = y_all[val_idx]

        train_ds = SectorSentimentDataset(train_texts, y_train, tokenizer, int(hparams["max_len"]))
        val_ds = SectorSentimentDataset(val_texts, y_val, tokenizer, int(hparams["max_len"]))

        class_weights = compute_class_weights_per_sector(y_train, num_classes)

        model = ModernBERTMultiHead(
            model_id=CONFIG["model_id"],
            num_sectors=num_sectors,
            num_classes=num_classes,
            dropout=float(hparams["dropout"]),
        ).to(device)

        args = TrainingArguments(
            output_dir=os.path.join(fold_dir, "hf_out"),
            learning_rate=float(hparams["lr"]),
            weight_decay=float(hparams["weight_decay"]),
            num_train_epochs=int(hparams["epochs"]),
            per_device_train_batch_size=int(hparams["train_bs"]),
            per_device_eval_batch_size=int(hparams["eval_bs"]),
            fp16=bool(CONFIG["fp16"] and torch.cuda.is_available()),
        
            eval_strategy=CONFIG["eval_strategy"],
            eval_steps=int(CONFIG["eval_steps"]),
            logging_steps=int(CONFIG["logging_steps"]),
        
            save_strategy="no",              # keep: no checkpoint saving
            metric_for_best_model="eval_macro_f1_mean",  # must match the eval metric name
            greater_is_better=True,
            report_to=[],
        )

        trainer = WeightedMultiHeadTrainer(
            class_weights=class_weights,
            model=model,
            args=args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            data_collator=collator,
            compute_metrics=compute_metrics,
            callbacks=[],  # or omit callbacks=
        )

        train_out = trainer.train()

        # Evaluate train + val
        train_metrics = trainer.evaluate(train_ds, metric_key_prefix="train")
        val_metrics = trainer.evaluate(val_ds, metric_key_prefix="val")

        save_json(train_out.metrics, os.path.join(fold_dir, "train_trainloop_metrics.json"))
        save_json(train_metrics, os.path.join(fold_dir, "train_metrics.json"))
        save_json(val_metrics, os.path.join(fold_dir, "val_metrics.json"))

        # Predictions
        pred_train, true_train = predict_ids(trainer, train_ds, num_classes)
        pred_val, true_val = predict_ids(trainer, val_ds, num_classes)

        # Per-sector JSON
        save_per_sector_train_val_json(
            out_dir=os.path.join(fold_dir, "per_sector"),
            sector_cols=sector_cols,
            sentiments=sentiments,
            y_true_train=true_train,
            y_pred_train=pred_train,
            y_true_val=true_val,
            y_pred_val=pred_val,
        )

        # Global artifacts
        save_global_fold_artifacts(
            fold_dir=os.path.join(fold_dir, "global"),
            y_true_train=true_train,
            y_pred_train=pred_train,
            y_true_val=true_val,
            y_pred_val=pred_val,
            target_names=sentiments,
        )

        # Curves
        tr_curve, va_curve = extract_loss_curves(trainer.state.log_history)
        tr_curve.to_csv(os.path.join(fold_dir, "train_loss_curve.csv"), index=False)
        va_curve.to_csv(os.path.join(fold_dir, "val_loss_curve.csv"), index=False)
        fold_train_curves.append(tr_curve)
        fold_val_curves.append(va_curve)

        fold_rows.append({
            "fold": fold_i,
            "train_macro_f1_mean": float(train_metrics.get("train_macro_f1_mean", np.nan)),
            "train_acc_mean": float(train_metrics.get("train_acc_mean", np.nan)),
            "train_loss": float(train_metrics.get("train_loss", np.nan)),
            "val_macro_f1_mean": float(val_metrics.get("val_macro_f1_mean", np.nan)),
            "val_acc_mean": float(val_metrics.get("val_acc_mean", np.nan)),
            "val_loss": float(val_metrics.get("val_loss", np.nan)),
        })

    # Summary
    summary = {
        "hparams": hparams,
        "hparams_hash": hp_hash,
        "n_folds_requested": CONFIG["n_folds"],
        "n_folds_ran": len(fold_rows),
        "fold_metrics": fold_rows,
        "train_macro_f1_mean_cv": float(np.nanmean([r["train_macro_f1_mean"] for r in fold_rows])),
        "train_macro_f1_std_cv": float(np.nanstd([r["train_macro_f1_mean"] for r in fold_rows])),
        "val_macro_f1_mean_cv": float(np.nanmean([r["val_macro_f1_mean"] for r in fold_rows])),
        "val_macro_f1_std_cv": float(np.nanstd([r["val_macro_f1_mean"] for r in fold_rows])),
        "val_loss_mean_cv": float(np.nanmean([r["val_loss"] for r in fold_rows])),
        "val_loss_std_cv": float(np.nanstd([r["val_loss"] for r in fold_rows])),
    }

    # Learning curve aggregation
    train_agg = aggregate_curves(fold_train_curves, "train")
    val_agg = aggregate_curves(fold_val_curves, "val")
    train_agg.to_csv(os.path.join(run_dir, "learning_curve_train_mean_std.csv"), index=False)
    val_agg.to_csv(os.path.join(run_dir, "learning_curve_val_mean_std.csv"), index=False)

    plot_learning_curve(
        train_agg,
        val_agg,
        out_png=os.path.join(run_dir, "learning_curve_mean_std.png"),
        title=f"Learning curve mean±std | h={hp_hash}",
    )

    save_json(summary, os.path.join(run_dir, "cv_summary.json"))
    return summary

# -------------------------
# Example usage (notebook)
# -------------------------
def prepare_data(data_train: pd.DataFrame) -> Tuple[pd.DataFrame, np.ndarray, List[str], np.ndarray, Dict[str, int]]:
    text_col = CONFIG["text_col"]
    sector_cols = CONFIG["sector_cols"]
    sentiments = [s.lower() for s in CONFIG["sentiments"]]
    sent2id = {s: i for i, s in enumerate(sentiments)}

    missing = [c for c in [text_col] + sector_cols if c not in data_train.columns]
    if missing:
        raise ValueError(f"Missing columns in data_train: {missing}")

    df = data_train.copy()
    y_all = np.stack([df[c].map(sent2id).values for c in sector_cols], axis=1).astype(np.int64)
    texts_all = df[text_col].fillna("").astype(str).tolist()

    strat_keys = make_stratify_keys(y_all, CONFIG["n_folds"], num_classes=len(sentiments))
    return df, y_all, texts_all, strat_keys, sent2id

# Handpicked hparams
HANDPICKED_HPARAMS = [
    {
        "lr": 2e-5,
        "weight_decay": 0.01,
        "dropout": 0.10,
        "train_bs": 8,
        "eval_bs": 16,
        "epochs": 2,
        "max_len": 2048,
    }
]

# In your notebook:
df, y_all, texts_all, strat_keys, sent2id = prepare_data(data_train)
all_summaries = []
for i, hp in enumerate(HANDPICKED_HPARAMS):
    s = run_cv_once(df, y_all, texts_all, strat_keys, hp, run_name=f"run_{i:02d}_h{hparams_hash(hp)}")
    all_summaries.append(s)
save_json({"runs": all_summaries}, os.path.join(CONFIG["output_root"], "all_runs_summary.json"))
print("Saved outputs to:", CONFIG["output_root"])